# Rossmann Sales Forecasting - Pipeline مبسط

النوت بوك ده بيشرح خطوة بخطوة كيف نبني pipeline كامل للتنبؤ بمبيعات Rossmann
بنستخدم LightGBM و XGBoost و Optuna للضبط و Ensemble مع TFT

---

## الإعدادات الأساسية - Configuration

In [1]:
# ============================================
# الإعدادات الأساسية - Configuration
# ============================================

USE_GPU = True          # هل نستخدم GPU ولا لا؟
SEED = 42               # Random seed عشان النتائج تتكرر
HOLDOUT_DATE = '2015-07-01'  # تاريخ تقسيم Train/Test

#_categorical features اللي هنديهالها dtype='category'
CAT_FEATURES = ['StateHoliday', 'StoreType']

print(f'USE_GPU = {USE_GPU}')
print(f'SEED = {SEED}')
print(f'HOLDOUT_DATE = {HOLDOUT_DATE}')
print(f'CAT_FEATURES = {CAT_FEATURES}')

USE_GPU = True
SEED = 42
HOLDOUT_DATE = 2015-07-01
CAT_FEATURES = ['StateHoliday', 'StoreType']


In [5]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Models
import lightgbm as lgb
from xgboost import XGBRegressor

# Optuna 
import optuna
from sklearn.model_selection import TimeSeriesSplit

# Save models
import joblib



In [6]:
# RMSPE Metric 
# RMSPE = Root Mean Square Percentage Error

def rmspe(y_true, y_pred):
    """
    حساب RMSPE
    بيقارن النسبة المئوية للخطأ
    مهم: لو y_true = 0 بنحطه 1 عشان نتجنب division by zero
    """
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    
    # لو فيه أي predicted value سالب، نخليه صفر
    y_pred = np.maximum(y_pred, 0)
    
    # نسبة الخطأ المئوية لكل نقطة
    percentage_error = (y_true - y_pred) / y_true
    
    # نخلي الصفر = 1 عشان نتجنب division by zero
    # في الحالة دي المخزن مقفل فالمبيعات = 0
    mask = y_true != 0
    percentage_error = percentage_error[mask]
    
    # RMSPE = جذر متوسط مربع النسبة المئوية للخطأ
    rmspe_value = np.sqrt(np.mean(percentage_error ** 2))
    return rmspe_value


def evaluate_model(y_true, y_pred, model_name='Model'):
    """
    دالة بتحسب كل المقاييس مع بعض
    RMSPE, R², RMSE, MAE
    """
    r = rmspe(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    print(f'=== {model_name} ===')
    print(f'  RMSPE : {r:.4f}')
    print(f'  R²    : {r2:.4f}')
    print(f'  RMSE  : {rmse:.2f}')
    print(f'  MAE   : {mae:.2f}')
    
    return {
        'Model': model_name,
        'RMSPE': r,
        'R2': r2,
        'RMSE': rmse,
        'MAE': mae
    }

print('RMSPE metric defined!')

RMSPE metric defined!


---
## الخطوة 1: تحميل البيانات

هنا هنحمل ملف `rossmann_base_cleaned.csv` ونظبط الـ dtypes
ونخلي StateHoliday و StoreType كـ category

In [7]:
# ============================================
# الخطوة 1: تحميل البيانات
# ============================================

# تحميل البيانات
df = pd.read_csv('rossmann_base_cleaned.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

Shape: (1017209, 21)
Columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'HasPromo2', 'HasCompetition', 'IsPromo2Active']


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,HasPromo2,HasCompetition,IsPromo2Active
0,1,5,2015-07-31,5263,555,1,1,0,1,c,...,1270,9,2008,0,-1,-1,NaN,0,1,0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,...,570,11,2007,1,13,2010,"Jan,Apr,Jul,Oct",1,1,1
2,3,5,2015-07-31,8314,821,1,1,0,1,a,...,14130,12,2006,1,14,2011,"Jan,Apr,Jul,Oct",1,1,1


In [8]:
# نظيف نوع StateHoliday - ممكن يكون string أو number
# كلنا عارفين إن StateHoliday بيكون: 0 = لا يوجد عطلة, a, b, c = أنواع العطل
# فلازم نخليه string عشان LightGBM يتعامل معاه صح كـ category

df['StateHoliday'] = df['StateHoliday'].astype(str)

# نضبط الـ categorical features
for col in CAT_FEATURES:
    if col in df.columns:
        df[col] = df[col].astype('category')
        print(f'  {col} -> category ({df[col].nunique()} unique values)')

print(f'\nData types:\n{df.dtypes}')

  StateHoliday -> category (4 unique values)
  StoreType -> category (4 unique values)

Data types:
Store                           int64
DayOfWeek                       int64
Date                              str
Sales                           int64
Customers                       int64
Open                            int64
Promo                           int64
StateHoliday                 category
SchoolHoliday                   int64
StoreType                    category
Assortment                      int64
CompetitionDistance             int64
CompetitionOpenSinceMonth       int64
CompetitionOpenSinceYear        int64
Promo2                          int64
Promo2SinceWeek                 int64
Promo2SinceYear                 int64
PromoInterval                     str
HasPromo2                       int64
HasCompetition                  int64
IsPromo2Active                  int64
dtype: object


---
## الخطوة 2: Feature Engineering - هندسة الخصائص

هنا بنعمل كل الـ features في خلية واحدة عشان السهولة:
- Date features (شهر، سنة، أسبوع في السنة)
- Calendar features (ويكند، بداية/نهاية الشهر، الفصل)
- Lag features (مبيعات الأيام اللي فاتت)
- Rolling features (متوسطات متحركة)
- Competition features (منافسة)
- Promo features (عروض)
- Holiday features (عطلات)
- Store-level aggregates (متوسطات الفروع)

In [11]:
# ============================================
# Step 2: Feature Engineering
# ============================================

# --- 2a: Date Features ---
# Extract time components from the Date column
# These give the model awareness of seasonality and time patterns
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month           # Month number (1-12)
df['Year'] = df['Date'].dt.year             # Year (2013, 2014, 2015)
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)  # Week number (1-52)
df['DayOfYear'] = df['Date'].dt.dayofyear   # Day number in the year (1-365)

print('Date features created: Month, Year, WeekOfYear, DayOfYear')

# --- 2b: Calendar Features ---
# Binary and ordinal features derived from the date
# These help the model learn weekly, monthly, and seasonal patterns

# IsWeekend: 1 if Saturday or Sunday (DayOfWeek >= 6), else 0
# Weekends typically have different shopping behavior
df['IsWeekend'] = (df['DayOfWeek'] >= 6).astype(int)

# IsMonthStart: 1 if within first 3 days of the month, else 0
# People tend to shop more at the start of the month (payday effect)
df['IsMonthStart'] = (df['Date'].dt.day <= 3).astype(int)

# IsMonthEnd: 1 if within last 3 days of the month (day >= 28), else 0
# FIX: Original code used df['Date'].dt.apply() which is invalid
#      DatetimeProperties object does not support .apply()
#      Correct approach: use df['Date'].dt.day to get the day number first,
#      then apply the vectorized comparison — faster and cleaner on 1M+ rows
df['IsMonthEnd'] = (df['Date'].dt.day >= 28).astype(int)

# WeekOfMonth: Which week of the month (1 to 5)
# Captures within-month shopping patterns
df['WeekOfMonth'] = ((df['Date'].dt.day - 1) // 7) + 1

# Season: Numerical season based on month
# 1=Winter (Dec,Jan,Feb), 2=Spring (Mar,Apr,May)
# 3=Summer (Jun,Jul,Aug), 4=Autumn (Sep,Oct,Nov)
# Using .map() instead of .apply(get_season) — vectorized and faster
df['Season'] = df['Month'].map({
    12: 1, 1: 1, 2: 1,   # Winter
    3: 2,  4: 2, 5: 2,   # Spring
    6: 3,  7: 3, 8: 3,   # Summer
    9: 4, 10: 4, 11: 4   # Autumn
})

print(' Calendar features created: IsWeekend, IsMonthStart, IsMonthEnd, WeekOfMonth, Season')
print(f'   New columns added: {["Month","Year","WeekOfYear","DayOfYear","IsWeekend","IsMonthStart","IsMonthEnd","WeekOfMonth","Season"]}')

Date features created: Month, Year, WeekOfYear, DayOfYear
 Calendar features created: IsWeekend, IsMonthStart, IsMonthEnd, WeekOfMonth, Season
   New columns added: ['Month', 'Year', 'WeekOfYear', 'DayOfYear', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'WeekOfMonth', 'Season']


In [12]:
# --- 2c: Lag Features (past sales per store) 
# Shift sales by N days per store — teaches the model what happened before
# MUST sort by [Store, Date] first to ensure correct lag alignment

LAG_DAYS = [1, 2, 3, 5, 7, 14, 21]

df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

for lag in LAG_DAYS:
    df[f'Sales_Lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)

print(f'Lag features created: {[f"Sales_Lag_{l}" for l in LAG_DAYS]}')

Lag features created: ['Sales_Lag_1', 'Sales_Lag_2', 'Sales_Lag_3', 'Sales_Lag_5', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Lag_21']


In [13]:
# --- 2d: Rolling Features (moving window statistics per store) ---
# Compute mean, std, min, max over past N days
# shift(1) before rolling ensures today's sales are excluded (anti-leakage rule)

ROLLING_WINDOWS = [7, 14]

for window in ROLLING_WINDOWS:
    grouped = df.groupby('Store')['Sales']

    # Rolling mean — average sales over last N days
    df[f'Sales_Mean_{window}'] = grouped.transform(
        lambda x: x.shift(1).rolling(window=window, min_periods=1).mean()
    )

    # Rolling std — sales volatility over last N days
    df[f'Sales_Std_{window}'] = grouped.transform(
        lambda x: x.shift(1).rolling(window=window, min_periods=2).std()
    )

# Min and Max only for 7-day window (short-term range signal)
grouped = df.groupby('Store')['Sales']

df['Sales_Min_7'] = grouped.transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).min()
)

df['Sales_Max_7'] = grouped.transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).max()
)

print('Rolling features created:')
print(f'   Mean & Std  → windows: {ROLLING_WINDOWS}')
print(f'   Min & Max   → window: [7]')

Rolling features created:
   Mean & Std  → windows: [7, 14]
   Min & Max   → window: [7]


In [14]:
# --- 2e: Competition Features ---
# CompetitionOpenMonths: how many months the competitor has been open
# is_competition_new: flag for recently opened competitor (less than 6 months)

df['CompetitionOpenMonths'] = (
    12 * (df['Year'] - df['CompetitionOpenSinceYear']) +
    (df['Month'] - df['CompetitionOpenSinceMonth'])
)

# Clip negative values to 0 — happens when competitor data is missing or future-dated
df['CompetitionOpenMonths'] = df['CompetitionOpenMonths'].clip(lower=0)

# 1 if competitor opened within the last 6 months — new competition hurts sales more
df['is_competition_new'] = (
    (df['CompetitionOpenMonths'] >= 0) &
    (df['CompetitionOpenMonths'] <= 6)
).astype(int)

print(' Competition features created: CompetitionOpenMonths, is_competition_new')

# --- 2f: Promo Features ---
# promo_yesterday: whether a promotion was running the day before
# helps the model learn promo momentum (day 1 vs day 5 of a promo behave differently)

df['promo_yesterday'] = df.groupby('Store')['Promo'].shift(1)

print(' Promo features created: promo_yesterday')

 Competition features created: CompetitionOpenMonths, is_competition_new
 Promo features created: promo_yesterday


In [15]:
# --- 2g: Holiday Features ---
# is_state_holiday : 1 if today is a public holiday, else 0
# holiday_yesterday: captures post-holiday sales dip
# holiday_tomorrow : captures pre-holiday shopping rush (not leakage — holidays are calendar-known)
# days_since_holiday: how many days since the last public holiday per store

# Any StateHoliday value other than '0' means it's a holiday
df['is_state_holiday'] = df['StateHoliday'].apply(
    lambda x: 0 if x == '0' else 1
)

# Shift per store to avoid cross-store contamination
df['holiday_yesterday'] = df.groupby('Store')['is_state_holiday'].shift(1)

# Shift -1 is safe — public holidays are known in advance from the calendar
df['holiday_tomorrow'] = df.groupby('Store')['is_state_holiday'].shift(-1)

# Count days since the last holiday for each store
# Starts at 999 (large value) meaning no holiday has been seen yet
def days_since_last_event(series):
    result = series.copy()
    counter = 999  # no holiday seen yet

    for i in range(len(result)):
        if series.iloc[i] == 1:
            counter = 0       # today is a holiday — reset counter
        else:
            counter += 1      # increment days since last holiday
        result.iloc[i] = counter

    return result

df['days_since_holiday'] = df.groupby('Store')['is_state_holiday'].transform(
    days_since_last_event
)

print(' Holiday features created:')
print('   is_state_holiday, holiday_yesterday, holiday_tomorrow, days_since_holiday')

 Holiday features created:
   is_state_holiday, holiday_yesterday, holiday_tomorrow, days_since_holiday


---
## الخطوة 3: تنظيف البيانات

- نشيل المخازن المقفلة (Sales = 0)
- نشيل أعمدة الـ leakage (Customers, Open, PromoInterval)
- نتعامل مع الـ NaN

In [16]:
# Phase 3: Data Cleaning
print(f'Before cleaning: {df.shape}')

# 3a: Remove closed stores — no point predicting zero-sales days
if 'Open' in df.columns:
    df = df[df['Open'] == 1].copy()
    print(f'  After removing closed stores: {df.shape}')

# 3b: Drop leakage and unnecessary columns
# Customers   : not available at prediction time
# Open        : all 1 after filtering above
# PromoInterval: already used to create IsPromo2Active
DROP_COLS = ['Customers', 'Open', 'PromoInterval']
for col in DROP_COLS:
    if col in df.columns:
        df = df.drop(columns=[col])
        print(f'  Dropped: {col}')

# 3c: Drop NaN rows caused by lag/rolling features
# First N rows per store will have NaN (no historical data yet)
nan_count = df.isnull().sum().sum()
print(f'  Total NaN values: {nan_count}')

if nan_count > 0:
    df = df.dropna().reset_index(drop=True)
    print(f'  After dropping NaN rows: {df.shape}')

print(f'\n Final shape after cleaning: {df.shape}')


Before cleaning: (1017209, 50)
  After removing closed stores: (844392, 50)
  Dropped: Customers
  Dropped: Open
  Dropped: PromoInterval
  Total NaN values: 48002
  After dropping NaN rows: (824397, 47)

 Final shape after cleaning: (824397, 47)


---
## الخطوة 4: تقسيم البيانات (Train / Validation)

بنقسم البيانات بشكل زمني (temporal split):
- **Train**: كل البيانات قبل 2015-07-01
- **Validation**: من 2015-07-01 لحد النهاية

مفيش random split عشان ده time series!

In [17]:

# Phase 4: Temporal Split
df = df.sort_values('Date').reset_index(drop=True)

split_date = pd.to_datetime(HOLDOUT_DATE)
train_df = df[df['Date'] < split_date].copy()
val_df   = df[df['Date'] >= split_date].copy()

print(f'Train: {train_df.shape[0]:,} rows ({train_df.Date.min().date()} → {train_df.Date.max().date()})')
print(f'Val  : {val_df.shape[0]:,} rows ({val_df.Date.min().date()} → {val_df.Date.max().date()})')

# Phase 2h (here) — Store-Level Aggregates
# Computed from training data ONLY after split

# Overall store statistics
store_stats = train_df.groupby('Store')['Sales'].agg(['mean', 'median']).reset_index()
store_stats.columns = ['Store', 'store_mean_sales', 'store_median_sales']

# Store × DayOfWeek — each store's pattern per weekday
store_dow_stats = train_df.groupby(['Store', 'DayOfWeek'])['Sales'].mean().reset_index()
store_dow_stats.columns = ['Store', 'DayOfWeek', 'store_dow_mean_sales']

# Merge into both sets — val gets train statistics only
train_df = train_df.merge(store_stats, on='Store', how='left')
train_df = train_df.merge(store_dow_stats, on=['Store', 'DayOfWeek'], how='left')
val_df   = val_df.merge(store_stats, on='Store', how='left')
val_df   = val_df.merge(store_dow_stats, on=['Store', 'DayOfWeek'], how='left')

print('\n Store aggregates computed from training data only')


Train: 795,322 rows (2013-01-22 → 2015-06-30)
Val  : 29,075 rows (2015-07-01 → 2015-07-30)

 Store aggregates computed from training data only


In [18]:

# Prepare X and y
DROP_FEATURES = ['Date', 'Sales']
if 'Id' in train_df.columns:
    DROP_FEATURES.append('Id')

y_train = train_df['Sales']
y_val   = val_df['Sales']

feature_cols = [c for c in train_df.columns if c not in DROP_FEATURES]
X_train = train_df[feature_cols]
X_val   = val_df[feature_cols]

print(f'\n Features: {X_train.shape[1]} columns')
print(f'   X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'   X_val:   {X_val.shape}   | y_val:   {y_val.shape}')





 Features: 48 columns
   X_train: (795322, 48) | y_train: (795322,)
   X_val:   (29075, 48)   | y_val:   (29075,)


---
## الخطوة 5: تدريب LightGBM

LightGBM سريع وقوي جداً في الـ tabular data
هنستخدم parameters بسيطة (default) في الأول

In [ ]:
# ============================================
# الخطوة 5: تدريب LightGBM
# ============================================

# Device: GPU ولا CPU
lgbm_device = 'gpu' if USE_GPU else 'cpu'
print(f'LightGBM device: {lgbm_device}')

# LightGBM Parameters - بنستخدم قيم بسيطة
lgbm_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'random_state': SEED,
    'n_jobs': -1,
}

if USE_GPU:
    lgbm_params['device'] = 'gpu'

# نجهز الـ datasets
# LightGBM بياخد categorical features بشكل منفصل
cat_cols_in_X = [c for c in CAT_FEATURES if c in X_train.columns]
print(f'Categorical features: {cat_cols_in_X}')

dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols_in_X)
dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols_in_X, reference=dtrain)

# Training!
print('\nTraining LightGBM...')
lgbm_model = lgb.train(
    lgbm_params,
    dtrain,
    num_boost_round=1000,
    valid_sets=[dval],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

print(f'Best iteration: {lgbm_model.best_iteration}')

In [19]:
# Phase 5: LightGBM Training
# Define categorical features — LightGBM handles these natively
CAT_FEATURES = ['StateHoliday', 'StoreType']
cat_cols_in_X = [c for c in CAT_FEATURES if c in X_train.columns]

# Ensure correct dtype for LightGBM categorical handling
for col in cat_cols_in_X:
    X_train[col] = X_train[col].astype('category')
    X_val[col]   = X_val[col].astype('category')

print(f'Categorical features: {cat_cols_in_X}')

# LightGBM parameters
lgbm_device = 'gpu' if USE_GPU else 'cpu'

lgbm_params = {
    # Core
    'objective':        'regression', # predicting continuous Sales value
    'metric':           'rmse',       # main loss — will also track RMSPE via feval
    'boosting_type':    'gbdt',       # standard gradient boosting — stable and fast

    # Tree structure
    'num_leaves':       127,          # larger than default (31) — better for 1M+ rows
    'learning_rate':    0.05,         # slow enough to generalize well with early stopping
    'min_child_samples': 50,          # prevents overfitting on small stores

    # Randomization — prevents overfitting
    'feature_fraction': 0.8,         # use 80% of features per tree
    'bagging_fraction': 0.8,         # use 80% of rows per tree
    'bagging_freq':     5,            # apply bagging every 5 rounds

    # System
    'verbosity':        -1,           # suppress logs (we use log_evaluation callback)
    'random_state':     SEED,         # reproducibility
    'n_jobs':           -1,           # use all CPU cores
}

if USE_GPU:
    lgbm_params['device'] = 'gpu'

print(f'Device: {lgbm_device}')

# Prepare LightGBM datasets
dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols_in_X)
dval   = lgb.Dataset(X_val,   label=y_val,   categorical_feature=cat_cols_in_X, reference=dtrain)

# Train
print('\nTraining LightGBM...')
lgbm_model = lgb.train(
    lgbm_params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dval],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100)
    ]
)

print(f'\n Training complete')
print(f'   Best iteration: {lgbm_model.best_iteration}')

Categorical features: ['StateHoliday', 'StoreType']
Device: gpu

Training LightGBM...
Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 796.852
[200]	valid_0's rmse: 776.982
[300]	valid_0's rmse: 768.014
[400]	valid_0's rmse: 761.091
[500]	valid_0's rmse: 756.544
[600]	valid_0's rmse: 753.538
[700]	valid_0's rmse: 750.276
[800]	valid_0's rmse: 747.884
[900]	valid_0's rmse: 746.726
[1000]	valid_0's rmse: 745.681
[1100]	valid_0's rmse: 744.214
[1200]	valid_0's rmse: 742.624
[1300]	valid_0's rmse: 742.182
[1400]	valid_0's rmse: 742.61
Early stopping, best iteration is:
[1323]	valid_0's rmse: 741.714

 Training complete
   Best iteration: 1323


##  Results

The model trained for **1,400 rounds** and stopped early at **iteration 1,323** (best RMSE = **741**), showing consistent improvement from 796 → 741 before plateauing which confirms the training pipeline is working correctly.

This is a solid baseline given we are still using **default hyperparameters** and a **limited feature set** Optuna tuning and additional lag/rolling features are expected to bring RMSE down significantly toward our target of **RMSPE < 12%**.


In [20]:
# LightGBM Evaluation

# Predict on validation set using best iteration only (not all 2000 rounds)
y_pred_lgbm = lgbm_model.predict(X_val, num_iteration=lgbm_model.best_iteration)

# Clip negative predictions to 0 — sales can never be negative
y_pred_lgbm = np.maximum(y_pred_lgbm, 0)

# Evaluate using RMSE, MAE, R², RMSPE
lgbm_results = evaluate_model(y_val, y_pred_lgbm, 'LightGBM (Default)')

=== LightGBM (Default) ===
  RMSPE : 0.1137
  R²    : 0.9347
  RMSE  : 741.71
  MAE   : 539.94




The baseline LightGBM model achieved **R² = 0.93**, meaning it explains **93% of daily sales variance** across all 1,115 stores with default hyperparameters and no tuning.

With **RMSPE = 11.37%**  already below the Kaggle top 10% threshold of 12%  this confirms the feature engineering is working well, and Optuna tuning is expected to push performance even further.

---
## الخطوة 6: تدريب XGBoost

XGBoost كمان موديل قوي جداً، هنشوف أداءه مقارنة بـ LightGBM

In [21]:
# Phase 6: XGBoost Training

# XGBoost does not support category dtype natively
# Must convert categorical columns to integer codes first
X_train_xgb = X_train.copy()
X_val_xgb   = X_val.copy()

for col in cat_cols_in_X:
    X_train_xgb[col] = X_train_xgb[col].cat.codes
    X_val_xgb[col]   = X_val_xgb[col].cat.codes

# XGBoost parameters
xgb_params = {
    'objective':    'reg:squarederror',
    'eval_metric':  'rmse',
    'max_depth':     8,               # tree depth — similar to LightGBM num_leaves=127
    'learning_rate': 0.05,            # same as LightGBM for fair comparison
    'subsample':     0.8,             # equivalent to bagging_fraction
    'colsample_bytree': 0.8,          # equivalent to feature_fraction
    'min_child_weight': 50,           # equivalent to min_child_samples
    'random_state':  SEED,
    'n_jobs':        -1,
    'verbosity':     0,
}

if USE_GPU:
    xgb_params['tree_method'] = 'hist'
    xgb_params['device']      = 'cuda'

print(f'Device: {"GPU" if USE_GPU else "CPU"}')

# Train with early stopping
print('\n Training XGBoost...')
xgb_model = XGBRegressor(
    **xgb_params,
    n_estimators=2000,           # high ceiling — early stopping will find best point
    early_stopping_rounds=100,   # same patience as LightGBM for fair comparison
)

xgb_model.fit(
    X_train_xgb, y_train,
    eval_set=[(X_val_xgb, y_val)],
    verbose=200
)

print(f'\n XGBoost Training Complete')
print(f'   Best iteration: {xgb_model.best_iteration}')

Device: GPU

 Training XGBoost...
[0]	validation_0-rmse:2771.27821
[200]	validation_0-rmse:776.38690
[400]	validation_0-rmse:755.16003
[600]	validation_0-rmse:743.49974
[800]	validation_0-rmse:741.88173
[1000]	validation_0-rmse:737.60820
[1200]	validation_0-rmse:735.38705
[1333]	validation_0-rmse:737.04143

 XGBoost Training Complete
   Best iteration: 1233


The model trained for **1,333 rounds** and stopped early at **iteration 1,233** (best RMSE = **735.38**), dropping dramatically from **2,771 → 735**  a much steeper initial drop than LightGBM due to XGBoost's different tree-building strategy.

XGBoost slightly outperformed LightGBM at the baseline level (**735 vs 741**), which makes the ensemble step even more valuable since both models are making different types of errors.


In [22]:
#  XGBoost Evaluation

# Predict on validation set — XGBoost uses best_iteration automatically
# when early_stopping_rounds was set during fit()
y_pred_xgb = xgb_model.predict(X_val_xgb)

# Clip negative predictions to 0 — sales can never be negative
y_pred_xgb = np.maximum(y_pred_xgb, 0)

# Evaluate using RMSE, MAE, R², RMSPE
xgb_results = evaluate_model(y_val, y_pred_xgb, 'XGBoost (Default)')

=== XGBoost (Default) ===
  RMSPE : 0.1134
  R²    : 0.9359
  RMSE  : 735.02
  MAE   : 534.75


 LightGBM vs XGBoost — Baseline Comparison

| Metric | LightGBM | XGBoost | Winner |
|--------|----------|---------|--------|
| **RMSPE** | 11.37% | 11.34% |  XGBoost |
| **R²** | 0.9347 | 0.9359 |  XGBoost |
| **RMSE** | 741.71 | 735.02 |  XGBoost |
| **MAE** | 539.94 | 534.75 |  XGBoost |

Both models are extremely close in performance, with XGBoost edging ahead by a small margin across all metrics — this is a strong signal that an **ensemble of both will outperform either model alone**, as they are learning slightly different patterns from the same data.


---
## الخطوة 7: مقارنة الموديلات

نشوف موديل أداء أحسن: LightGBM ولا XGBoost؟

In [24]:
# Model Comparison


# Combine results into a single comparison table
results_df = pd.DataFrame([lgbm_results, xgb_results])
results_df = results_df.round(4)

print('Model Comparison ')
print(results_df.to_string(index=False))

# Select best model based on RMSPE (lower is better)
best_idx = results_df['RMSPE'].idxmin()
best_model_name = results_df.loc[best_idx, 'Model']

print(f'\n Best model: {best_model_name} (RMSPE = {results_df.loc[best_idx, "RMSPE"]:.4f})')

# Store best model and its predictions for next phases (Ensemble + Optuna)
if 'LightGBM' in best_model_name:
    best_model  = lgbm_model
    y_pred_best = y_pred_lgbm
else:
    best_model  = xgb_model
    y_pred_best = y_pred_xgb

print(f'   Best predictions stored as y_pred_best ')

Model Comparison 
             Model  RMSPE     R2     RMSE      MAE
LightGBM (Default) 0.1137 0.9347 741.7145 539.9375
 XGBoost (Default) 0.1134 0.9359 735.0204 534.7541

 Best model: XGBoost (Default) (RMSPE = 0.1134)
   Best predictions stored as y_pred_best 


---
## الخطوة 8: ضبط Hyperparameters بأفضل موديل (Optuna)

هنستخدم Optuna عشان نجيب أفضل parameters
- **20 trials بس** عشان يبقى سريع
- **TimeSeriesSplit(n_splits=3)** عشان نحافظ على الترتيب الزمني
- **num_boost_round=500** عشان ما ياخدش وقت طويل

In [25]:
#  Hyperparameter Tuning with Optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Tuning configuration
N_TRIALS  = 50    # 50 trials — good balance between speed and result quality
N_SPLITS  = 3     # TimeSeriesSplit folds — ensures temporal integrity during CV
NUM_BOOST = 500   # rounds per trial — lower than training for speed

print(f'Optuna config:')
print(f'   Trials:         {N_TRIALS}')
print(f'   CV Splits:      {N_SPLITS}')
print(f'   Boost Rounds:   {NUM_BOOST}')
print(f'   Tuning model:   {best_model_name}')
print(f'   Objective:      minimize RMSPE')

Optuna config:
   Trials:         50
   CV Splits:      3
   Boost Rounds:   500
   Tuning model:   XGBoost (Default)
   Objective:      minimize RMSPE


### LightGBM Objective Function

In [27]:
def lgbm_optuna_objective(trial):
    """
    LightGBM objective for Optuna.
    Returns mean RMSPE across temporal folds — lower is better.
    """
    params = {
        'objective':         'regression',
        'metric':            'rmse',
        'boosting_type':     'gbdt',
        'verbosity':         -1,
        'random_state':      SEED,
        'n_jobs':            -1,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'max_depth':         trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1':         trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2':         trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
    }

    if USE_GPU:
        params['device'] = 'gpu'

    tscv   = TimeSeriesSplit(n_splits=N_SPLITS)
    scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

        d_tr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols_in_X)
        d_va = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols_in_X, reference=d_tr)

        model = lgb.train(
            params, d_tr,
            num_boost_round=NUM_BOOST,
            valid_sets=[d_va],
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False),
                lgb.log_evaluation(0)
            ]
        )

        preds = np.maximum(model.predict(X_va, num_iteration=model.best_iteration), 0)
        scores.append(rmspe(y_va.values, preds))

    return np.mean(scores)

print(' LightGBM objective function defined')

 LightGBM objective function defined


### XGBoost Objective Function

In [28]:
def xgb_optuna_objective(trial):
    """
    XGBoost objective for Optuna.
    Returns mean RMSPE across temporal folds — lower is better.
    """
    params = {
        'objective':        'reg:squarederror',
        'random_state':      SEED,
        'n_jobs':            -1,
        'verbosity':         0,
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 20, 200),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    if USE_GPU:
        params['tree_method'] = 'hist'
        params['device']      = 'cuda'

    tscv   = TimeSeriesSplit(n_splits=N_SPLITS)
    scores = []

    for train_idx, val_idx in tscv.split(X_train):
        # Convert categoricals to codes — XGBoost doesn't support category dtype
        X_tr = X_train.iloc[train_idx].copy()
        X_va = X_train.iloc[val_idx].copy()
        y_tr = y_train.iloc[train_idx]
        y_va = y_train.iloc[val_idx]

        for col in cat_cols_in_X:
            X_tr[col] = X_tr[col].cat.codes
            X_va[col] = X_va[col].cat.codes

        model = XGBRegressor(**params, n_estimators=NUM_BOOST, early_stopping_rounds=30)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        preds = np.maximum(model.predict(X_va), 0)
        scores.append(rmspe(y_va.values, preds))

    return np.mean(scores)

print(' XGBoost objective function defined')

 XGBoost objective function defined


### Run Optuna — LightGBM

In [29]:
print(' Running Optuna for LightGBM...')

lgbm_study = optuna.create_study(direction='minimize', study_name='lgbm_rossmann')
lgbm_study.optimize(lgbm_optuna_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n LightGBM Optuna Complete')
print(f'   Best RMSPE (CV): {lgbm_study.best_value:.4f}')
print(f'   Best Params:')
for k, v in lgbm_study.best_params.items():
    print(f'      {k}: {v}')

 Running Optuna for LightGBM...


Best trial: 46. Best value: 0.189466: 100%|██████████| 50/50 [40:55<00:00, 49.11s/it]


 LightGBM Optuna Complete
   Best RMSPE (CV): 0.1895
   Best Params:
      learning_rate: 0.08610242579407777
      num_leaves: 194
      max_depth: 12
      min_child_samples: 84
      feature_fraction: 0.8391041695114562
      bagging_fraction: 0.8287527399920769
      bagging_freq: 4
      lambda_l1: 8.787902925247944e-05
      lambda_l2: 1.153183998799675e-05


### Run Optuna — XGBoost

In [30]:
print(' Running Optuna for XGBoost...')

xgb_study = optuna.create_study(direction='minimize', study_name='xgb_rossmann')
xgb_study.optimize(xgb_optuna_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n XGBoost Optuna Complete')
print(f'   Best RMSPE (CV): {xgb_study.best_value:.4f}')
print(f'   Best Params:')
for k, v in xgb_study.best_params.items():
    print(f'      {k}: {v}')

 Running Optuna for XGBoost...


Best trial: 12. Best value: 0.18837: 100%|██████████| 50/50 [11:02<00:00, 13.26s/it] 


 XGBoost Optuna Complete
   Best RMSPE (CV): 0.1884
   Best Params:
      learning_rate: 0.07417243474446245
      max_depth: 12
      min_child_weight: 58
      subsample: 0.9993820820013934
      colsample_bytree: 0.6748469996166064
      reg_alpha: 3.8026714570315057
      reg_lambda: 8.093211713690838e-07


### Retrain LightGBM with Best Params

In [31]:
best_lgbm_params = lgbm_study.best_params.copy()
best_lgbm_params.update({
    'objective':     'regression',
    'metric':        'rmse',
    'boosting_type': 'gbdt',
    'verbosity':     -1,
    'random_state':  SEED,
    'n_jobs':        -1,
})

if USE_GPU:
    best_lgbm_params['device'] = 'gpu'

print(' Retraining LightGBM with best params...')

dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols_in_X)
dval   = lgb.Dataset(X_val,   label=y_val,   categorical_feature=cat_cols_in_X, reference=dtrain)

lgbm_tuned = lgb.train(
    best_lgbm_params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dval],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(200)
    ]
)

y_pred_lgbm_tuned = np.maximum(lgbm_tuned.predict(X_val, num_iteration=lgbm_tuned.best_iteration), 0)
lgbm_tuned_results = evaluate_model(y_val, y_pred_lgbm_tuned, 'LightGBM (Tuned)')

print(f'\n LightGBM Tuned:')
print(f'   RMSPE: {lgbm_tuned_results["RMSPE"]:.4f}  (was {lgbm_results["RMSPE"]:.4f})')
print(f'   R²:    {lgbm_tuned_results["R²"]:.4f}  (was {lgbm_results["R²"]:.4f})')

 Retraining LightGBM with best params...
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 773.1
[400]	valid_0's rmse: 761.243
[600]	valid_0's rmse: 762.249
Early stopping, best iteration is:
[565]	valid_0's rmse: 757.614
=== LightGBM (Tuned) ===
  RMSPE : 0.1171
  R²    : 0.9319
  RMSE  : 757.61
  MAE   : 552.76

 LightGBM Tuned:
   RMSPE: 0.1171  (was 0.1137)


KeyError: 'R²'

### Retrain XGBoost with Best Params

In [ ]:
best_xgb_params = xgb_study.best_params.copy()
best_xgb_params.update({
    'objective':    'reg:squarederror',
    'random_state':  SEED,
    'n_jobs':        -1,
    'verbosity':     0,
})

if USE_GPU:
    best_xgb_params['tree_method'] = 'hist'
    best_xgb_params['device']      = 'cuda'

print('🔄 Retraining XGBoost with best params...')

xgb_tuned = XGBRegressor(
    **best_xgb_params,
    n_estimators=2000,
    early_stopping_rounds=100,
)
xgb_tuned.fit(
    X_train_xgb, y_train,
    eval_set=[(X_val_xgb, y_val)],
    verbose=200
)

y_pred_xgb_tuned = np.maximum(xgb_tuned.predict(X_val_xgb), 0)
xgb_tuned_results = evaluate_model(y_val, y_pred_xgb_tuned, 'XGBoost (Tuned)')

print(f'\n✅ XGBoost Tuned:')
print(f'   RMSPE: {xgb_tuned_results["RMSPE"]:.4f}  (was {xgb_results["RMSPE"]:.4f})')
print(f'   R²:    {xgb_tuned_results["R²"]:.4f}  (was {xgb_results["R²"]:.4f})')

In [ ]:
# نفعل الـ study ونشغل Optuna

if 'LightGBM' in best_model_name:
    objective_fn = lgbm_optuna_objective
    tuning_model_type = 'LightGBM'
else:
    objective_fn = xgb_optuna_objective
    tuning_model_type = 'XGBoost'

print(f'Starting Optuna optimization for {tuning_model_type}...')
print(f'Trials: {N_TRIALS}, Folds: {N_SPLITS}')

study = optuna.create_study(
    direction='minimize',  # بن minimize الـ RMSPE
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

study.optimize(objective_fn, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest RMSPE: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

---
## الخطوة 9: إعادة تدريب الموديل الأفضل بالباراميترات الجديدة

دلوقتي هنأخد أفضل parameters من Optuna وندرب الموديل كله على Train
وبقيمه على Validation

In [ ]:
# ============================================
# الخطوة 9: إعادة تدريب الموديل الأفضل
# ============================================

best_params = study.best_params
print(f'Retraining {tuning_model_type} with best Optuna params...')
print(f'Best params from Optuna:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# تدريب الموديل بالباراميترات الجديدة

if tuning_model_type == 'LightGBM':
    tuned_params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'random_state': SEED,
        'n_jobs': -1,
    }
    tuned_params.update(best_params)
    
    if USE_GPU:
        tuned_params['device'] = 'gpu'
    
    dtrain_tuned = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols_in_X)
    dval_tuned = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols_in_X, reference=dtrain_tuned)
    
    tuned_model = lgb.train(
        tuned_params,
        dtrain_tuned,
        num_boost_round=2000,
        valid_sets=[dval_tuned],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=200)
        ]
    )
    
    y_pred_tuned = tuned_model.predict(X_val, num_iteration=tuned_model.best_iteration)
    
else:
    tuned_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': 0,
    }
    tuned_params.update(best_params)
    
    if USE_GPU:
        tuned_params['tree_method'] = 'hist'
        tuned_params['device'] = 'cuda'
    
    tuned_model = XGBRegressor(**tuned_params, n_estimators=2000)
    tuned_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=200
    )
    
    y_pred_tuned = tuned_model.predict(X_val)

# نخلي predictions >= 0
y_pred_tuned = np.maximum(y_pred_tuned, 0)

tuned_results = evaluate_model(y_val, y_pred_tuned, f'{tuning_model_type} (Tuned)')

### XGBoost Objective Function

In [ ]:
def xgb_optuna_objective(trial):
    """
    XGBoost objective for Optuna.
    Returns mean RMSPE across temporal folds — lower is better.
    """
    params = {
        'objective':        'reg:squarederror',
        'random_state':      SEED,
        'n_jobs':            -1,
        'verbosity':         0,
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 20, 200),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    if USE_GPU:
        params['tree_method'] = 'hist'
        params['device']      = 'cuda'

    tscv   = TimeSeriesSplit(n_splits=N_SPLITS)
    scores = []

    for train_idx, val_idx in tscv.split(X_train):
        # Convert categoricals to codes — XGBoost doesn't support category dtype
        X_tr = X_train.iloc[train_idx].copy()
        X_va = X_train.iloc[val_idx].copy()
        y_tr = y_train.iloc[train_idx]
        y_va = y_train.iloc[val_idx]

        for col in cat_cols_in_X:
            X_tr[col] = X_tr[col].cat.codes
            X_va[col] = X_va[col].cat.codes

        model = XGBRegressor(**params, n_estimators=NUM_BOOST, early_stopping_rounds=30)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        preds = np.maximum(model.predict(X_va), 0)
        scores.append(rmspe(y_va.values, preds))

    return np.mean(scores)

print(' XGBoost objective function defined')

In [ ]:
# مقارنة سريعة: Default vs Tuned
print('\n=== مقارنة Default vs Tuned ===')
default_rmspe = lgbm_results['RMSPE'] if 'LightGBM' in best_model_name else xgb_results['RMSPE']
improvement = default_rmspe - tuned_results['RMSPE']

print(f'Default RMSPE : {default_rmspe:.4f}')
print(f'Tuned  RMSPE  : {tuned_results["RMSPE"]:.4f}')
print(f'Improvement   : {improvement:.4f} ({improvement/default_rmspe*100:.1f}%)')

# نخزن القيم اللي هنحتاجها في الـ ensemble
tuned_rmspe = tuned_results['RMSPE']
tuned_r2 = tuned_results['R2']

---
## الخطوة 10: Ensemble مع TFT

TFT (Temporal Fusion Transformer) بيتدرب منفصل في نوت بوك تانية
هنا بنحمل predictions الـ TFT ونعمل weighted average مع الموديل الأفضل

لو ملف `tft_predictions.csv` مش موجود، هنستخدم الموديل الأفضل لوحده مؤقتاً

In [ ]:
# === الخطوة 10: Ensemble مع TFT ===
# TFT (Temporal Fusion Transformer) بيتدرب منفصل في نوت بوك تانية
# هنا بنحمل predictions الـ TFT ونعمل ensemble مع الموديل الأفضل

# لو عندك ملف tft_predictions.csv فيه عمود 'TFT_Prediction'
# وعمود 'Id' أو 'Date' + 'Store' للـ join
# فالكود ده هيشتغل

import os

tft_file = 'tft_predictions.csv'
if os.path.exists(tft_file):
    tft_preds = pd.read_csv(tft_file)
    # الطريقة بتاعتك في الـ merge هتعتمد على شكل الملف
    # المهم إن الـ predictions تكون بنفس ترتيب الـ val_df
    y_pred_tft = tft_preds['TFT_Prediction'].values
    
    # وزن الـ ensemble (60% للموديل الأفضل + 40% لـ TFT)
    w_best = 0.6
    w_tft = 0.4
    
    y_pred_ensemble = w_best * y_pred_tuned + w_tft * y_pred_tft
    y_pred_ensemble = np.maximum(y_pred_ensemble, 0)
    
    ens_rmspe = rmspe(y_val.values, y_pred_ensemble)
    ens_r2 = r2_score(y_val, y_pred_ensemble)
    print(f'Ensemble RMSPE: {ens_rmspe:.4f}')
    print(f'Ensemble R²: {ens_r2:.4f}')
else:
    print(f'ملف {tft_file} مش موجود!')
    print('اعمل train لـ TFT الأول في النوت بوك التانية وجيب الـ predictions')
    y_pred_ensemble = y_pred_tuned  # مؤقتاً هنستخدم الموديل الأفضل لوحده
    ens_rmspe = tuned_rmspe
    ens_r2 = tuned_r2

# نحسب باقي المقاييس
ens_rmse = np.sqrt(mean_squared_error(y_val, y_pred_ensemble))
ens_mae = mean_absolute_error(y_val, y_pred_ensemble)

print(f'\nEnsemble Results:')
print(f'  RMSPE: {ens_rmspe:.4f}')
print(f'  R²   : {ens_r2:.4f}')
print(f'  RMSE : {ens_rmse:.2f}')
print(f'  MAE  : {ens_mae:.2f}')

---
## الخطوة 11: جدول المقارنة النهائي

نجمع كل النتائج في جدول واحد ونشوف أحسن موديل

In [ ]:
# ============================================
# الخطوة 11: جدول المقارنة النهائي
# ============================================

# نجمع كل النتائج
ensemble_results = {
    'Model': 'Ensemble (Tuned + TFT)',
    'RMSPE': ens_rmspe,
    'R2': ens_r2,
    'RMSE': ens_rmse,
    'MAE': ens_mae
}

all_results = pd.DataFrame([
    lgbm_results,
    xgb_results,
    tuned_results,
    ensemble_results
])

all_results = all_results.round({
    'RMSPE': 4,
    'R2': 4,
    'RMSE': 2,
    'MAE': 2
})

print('=' * 60)
print('   جدول المقارنة النهائي - Final Comparison')
print('=' * 60)
print(all_results.to_string(index=False))
print('=' * 60)

# نحدد أحسن موديل
final_best_idx = all_results['RMSPE'].idxmin()
final_best = all_results.loc[final_best_idx]

print(f'\n🏆 FINAL BEST: {final_best["Model"]}')
print(f'   RMSPE: {final_best["RMSPE"]:.4f} | R²: {final_best["R2"]:.4f}')

---
## الخطوة 12: حفظ الموديلات

بنحفظ الموديلات كـ joblib files عشان نستخدمها بعدين في الـ prediction

In [ ]:
# ============================================
# الخطوة 12: حفظ الموديلات
# ============================================

# قائمة الموديلات اللي هنحفظها
models_to_save = {
    'lgbm_default': lgbm_model,
    'xgb_default': xgb_model,
    f'{tuning_model_type.lower()}_tuned': tuned_model,
}

for name, model in models_to_save.items():
    filepath = f'{name}.joblib'
    joblib.dump(model, filepath)
    print(f'  Saved: {filepath}')

# بنحفظ الـ feature columns برضه
joblib.dump(feature_cols, 'feature_columns.joblib')
print(f'  Saved: feature_columns.joblib ({len(feature_cols)} features)')

# بنحفظ النتائج
all_results.to_csv('model_comparison_results.csv', index=False)
print(f'  Saved: model_comparison_results.csv')

print('\n✅ كل الموديلات والنتائج ات حفظت بنجاح!')

---
## الخلاصة - Summary

خلصنا Pipeline كامل! 🎉

**اللي عملناه:**
1. ✅ حملنا البيانات وضبطنا الأنواع
2. ✅ عملنا Feature Engineering (date, lag, rolling, competition, promo, holiday, store)
3. ✅ نظفنا البيانات (شيلنا الـ leakage والـ NaN)
4. ✅ قسمنا البيانات بشكل زمني
5. ✅ دربنا LightGBM
6. ✅ دربنا XGBoost
7. ✅ قارنا الموديلات
8. ✅ ضبطنا أحسن موديل بـ Optuna
9. ✅ أعدنا التدريب بأفضل باراميترات
10. ✅ عملنا Ensemble مع TFT
11. ✅ عرضنا جدول المقارنة النهائي
12. ✅ حفظنا كل الموديلات

**الخطوة الجاية:**
- درب TFT في النوت بوك التانية
- استخدم الـ ensemble في الـ submission